In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -r /content/drive/MyDrive/isic_2024_second_place/input/isic2024-pip/requirements.txt
!pip install hydra-core lightning catboost rootutils

import hydra
import rootutils
import joblib
import os
from lightning import LightningDataModule
from hydra import compose, initialize_config_dir
from pathlib import Path
from glob import glob
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import gc

os.chdir("/content/drive/MyDrive/isic_2024_second_place/working/src")

rootutils.setup_root(Path().resolve(), indicator=".project-root", pythonpath=True)

from src.utils import RankedLogger
from src.isic_utils.feature_engineering import feature_engineering_new
from src.isic_utils.utils import preprocess_df

log = RankedLogger(__name__, rank_zero_only=True)

gbdt_params = "0906-9NNs-18types-feV7-s5-tuning_weights"
batch_size_pred = 128
DEBUG_WITH_TRAIN_DATA = False
average = "simple"
use_model_dnn = "all_data"
use_model_gbdt = "all_data"
use_shina_models = False

assert average in ["simple", "rank"]
assert use_model_dnn in ["kfold", "all_data"]
assert use_model_gbdt in ["kfold", "all_data"]

with initialize_config_dir(version_base=None, config_dir="/content/drive/MyDrive/isic_2024_second_place/working/configs"):
    cfg = compose(
        config_name="gbdt",
        overrides=[f"gbdt_params={gbdt_params}"],
        return_hydra_config=True,
    )
    cfg.paths.output_dir = "${hydra.runtime.output_dir}"
    cfg.paths.work_dir = "${hydra.runtime.cwd}"
    cfg.hydra.run.dir = cfg.log_dir
    cfg.hydra.runtime.output_dir = cfg.hydra.run.dir

df_train = pd.read_csv(os.path.join(cfg.data.data_dir, cfg.data.meta_csv_train_name))
df_test = pd.read_csv(os.path.join(cfg.data.data_dir, cfg.data.meta_csv_test_name))

df_train, feature_cols, cat_cols = feature_engineering_new(df_train, version=cfg.gbdt_params.version_fe)
df_test, _, _ = feature_engineering_new(df_test, version=cfg.gbdt_params.version_fe)
df_train, df_test, feature_cols, cat_cols = preprocess_df(df_train, df_test, feature_cols, cat_cols)

del df_train
gc.collect()

def strip_orig_mod_prefix(state_dict):
    new_state_dict = {}
    for key, value in state_dict.items():
        new_key = key.replace("net._orig_mod.", "net.")
        new_state_dict[new_key] = value
    return new_state_dict

def inference_all_data(temp_dir, cfg):
    print("Running inference on single device.")
    os.makedirs(temp_dir, exist_ok=True)

    for dnn_run in cfg.gbdt_params.dnn_predictions:
        config_path = os.path.join("/content/drive/MyDrive/isic_2024_second_place/working/configs/experiment", f"{dnn_run.name}.yaml")
        if not os.path.exists(config_path):
            print(f"Configuration file for {dnn_run.name} not found at {config_path}. Skipping this experiment.")
            continue

        with initialize_config_dir(version_base=None, config_dir="/content/drive/MyDrive/isic_2024_second_place/working/configs"):
            cfg_dnn = compose(
                config_name="train",
                overrides=[f"experiment={dnn_run.name}"],
                return_hydra_config=True,
            )
            cfg_dnn.task_name = "train_all_data"
            cfg_dnn.paths.output_dir = "${hydra.runtime.output_dir}"
            cfg_dnn.paths.work_dir = "${hydra.runtime.cwd}"
            cfg_dnn.hydra.run.dir = cfg_dnn.log_dir
            cfg_dnn.hydra.runtime.output_dir = cfg_dnn.hydra.run.dir

        if cfg_dnn.model.net.get("pretrained"):
            cfg_dnn.model.net.pretrained = False
        if cfg_dnn.model.net.get("my_pretrain_path"):
            cfg_dnn.model.net.my_pretrain_path = None
        if cfg_dnn.model.net.get("image_model"):
            if cfg_dnn.model.net.image_model.get("pretrained"):
                cfg_dnn.model.net.image_model.pretrained = False
            if cfg_dnn.model.net.image_model.get("my_pretrain_path"):
                cfg_dnn.model.net.image_model.my_pretrain_path = None
        if cfg_dnn.model.net.get("meta_pretrain_path"):
            cfg_dnn.model.net.meta_pretrain_path = None
        if cfg_dnn.model.net.get("image_pretrain_path"):
            cfg_dnn.model.net.image_pretrain_path = None
        if cfg_dnn.model.net.get("ckpt_path"):
            cfg_dnn.model.net.ckpt_path = None
        if cfg_dnn.model.net.get("encoder_image"):
            cfg_dnn.model.net.encoder_image.pretrained = False

        cfg_dnn.data.num_workers = 2

        datamodule: LightningDataModule = hydra.utils.instantiate(cfg_dnn.data)
        datamodule.setup("predict")
        dataset = datamodule.data_pred
        dataloader = DataLoader(dataset, batch_size=batch_size_pred, num_workers=2)

        ckpt_name = "last.ckpt"
        ckpt_path = glob(os.path.join(cfg_dnn.paths.output_dir, "checkpoints", ckpt_name))[0]

        model: LightningDataModule = hydra.utils.instantiate(cfg_dnn.model)
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {device}")
        model = model.to(device).eval()
        cfg_dnn.model.compile = False

        state_dict = torch.load(ckpt_path, map_location=device)["state_dict"]
        state_dict = strip_orig_mod_prefix(state_dict)
        model.load_state_dict(state_dict)
        net = model.net
        net.eval()

        all_predictions = []
        ids = []
        with torch.inference_mode():
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                for data in tqdm(dataloader, desc="single_device"):
                    if cfg_dnn.model._target_ == "src.models.isic2024_module_tip_finetune.ISIC2024LitModuleTIPFinetune":
                        logits = net(data["image"].to(device), data["tabular"].to(device))
                    else:
                        if cfg_dnn.model.get("use_image") and cfg_dnn.model.get("use_metadata"):
                            logits = net(data["image"].to(device), data["metadata_num"].to(device), data["metadata_cat"].to(device))
                        else:
                            logits = net(data["image"].to(device))

                    if cfg.gbdt_params.use_logits:
                        preds = logits[:, 1]
                    else:
                        preds = torch.softmax(logits, dim=1)[:, 1]
                    all_predictions.append(preds.cpu())
                    ids.append(data["isic_id"])

        all_predictions = torch.cat(all_predictions)
        ids = np.concatenate(ids)

        temp_file = os.path.join(temp_dir, f"predictions_{dnn_run.name}.pt")
        torch.save(all_predictions, temp_file)
        temp_file_id = os.path.join(temp_dir, f"ids_{dnn_run.name}.npy")
        np.save(temp_file_id, ids)

        print(f"{dnn_run.name} finished inference with predictions saved to {temp_file}, {temp_file_id}")

        del all_predictions, ids, datamodule, dataset, model, net, dataloader
        gc.collect()

def inference_mp_all_data(cfg):
    temp_dir = "/content/drive/MyDrive/isic_2024_second_place/tmp/inference_temp"
    print(f"Temporary directory for storing predictions: {temp_dir}")

    inference_all_data(temp_dir, cfg)

    dnn_run_name_list = []
    df_dnn_preds_list = []
    for dnn_run in cfg.gbdt_params.dnn_predictions:
        temp_file = os.path.join(temp_dir, f"predictions_{dnn_run.name}.pt")
        temp_file_id = os.path.join(temp_dir, f"ids_{dnn_run.name}.npy")
        if not os.path.exists(temp_file) or not os.path.exists(temp_file_id):
            print(f"Prediction files for {dnn_run.name} not found at {temp_file} or {temp_file_id}. Skipping this experiment.")
            continue

        try:
            predictions = torch.load(temp_file)
            ids = np.load(temp_file_id)
            if not isinstance(predictions, torch.Tensor) or not isinstance(ids, np.ndarray):
                print(f"Invalid data format for {dnn_run.name} at {temp_file} or {temp_file_id}. Skipping this experiment.")
                continue
            if len(predictions) == 0 or len(ids) == 0:
                print(f"Empty predictions or IDs for {dnn_run.name} at {temp_file} or {temp_file_id}. Skipping this experiment.")
                continue
            if len(predictions) != len(ids):
                print(f"Mismatch in lengths of predictions ({len(predictions)}) and IDs ({len(ids)}) for {dnn_run.name}. Skipping this experiment.")
                continue

            df_preds = pd.DataFrame({"isic_id": ids, "target": predictions})
            df_preds = df_preds.drop_duplicates(subset=["isic_id"])
            df_preds = df_preds.rename({"target": dnn_run.name}, axis=1)

            df_dnn_preds_list.append(df_preds)
            dnn_run_name_list.append(dnn_run.name)
        except Exception as e:
            print(f"Error loading or processing predictions for {dnn_run.name}: {str(e)}. Skipping this experiment.")
            continue

    return df_dnn_preds_list, dnn_run_name_list

df_dnn_preds_list, dnn_run_name_list = inference_mp_all_data(cfg)
for run_name, df_preds in zip(dnn_run_name_list, df_dnn_preds_list):
    df_test = df_test.merge(df_preds[["isic_id", run_name]], how="left", on="isic_id")

feature_cols += dnn_run_name_list

missing_dnn_features = [dnn_run.name for dnn_run in cfg.gbdt_params.dnn_predictions if dnn_run.name not in dnn_run_name_list]
for missing_feature in missing_dnn_features:
    if missing_feature not in df_test.columns:
        df_test[missing_feature] = 0.0
        print(f"Added placeholder column for missing DNN feature: {missing_feature}")

save_file_path = os.path.join(cfg.log_dir, f"model_all_data.joblib")
model = joblib.load(save_file_path)
target = model.predict(df_test)

del model
gc.collect()

assert not DEBUG_WITH_TRAIN_DATA
df_submit = pd.DataFrame({"isic_id": df_test["isic_id"], "target": target})
df_submit.to_csv("/content/drive/MyDrive/isic_2024_second_place/working/submission.csv", index=False)

from IPython.display import display
display(df_submit.head())

Mounted at /content/drive
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 139.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.2/164.2 MB 15.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 108.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 58.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 130.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 21.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 45.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-3031214054.py:66: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(os.path.join(cfg.data.data_dir, cfg.data.meta_csv_train_name))


Analyzing ducklings

Streaming output truncated to the last 5000 lines.
/content/drive/MyDrive/isic_2024_second_place/working/src/isic_utils/feature_engineering.py:2908: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  z_scores = group[ud_columns].apply(lambda x: zscore(x, nan_policy="omit"))
/content/drive/MyDrive/isic_2024_second_place/working/src/isic_utils/feature_engineering.py:2908: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  z_scores = group[ud_columns].apply(lambda x: zscore(x, nan_policy="omit"))
/content/drive/MyDrive/isic_2024_second_place/working/src/isic_utils/feature_engineering.py:2908: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be 


Concat ducklings
Extending ducklings
Enhancing ugly duckling features........................................................................................................
Analyzing ducklings

...

Using device: cuda:0


single_device:   0%|          | 0/1 [00:00<?, ?it/s]

0828-resnext50-transV2-lr1e-3-target_decay001-bs32_8-ep200-neg3-cluster7t5-tsgkf finished inference with predictions saved to /content/drive/MyDrive/isic_2024_second_place/tmp/inference_temp/predictions_0828-resnext50-transV2-lr1e-3-target_decay001-bs32_8-ep200-neg3-cluster7t5-tsgkf.pt, /content/drive/MyDrive/isic_2024_second_place/tmp/inference_temp/ids_0828-resnext50-transV2-lr1e-3-target_decay001-bs32_8-ep200-neg3-cluster7t5-tsgkf.npy
Configuration file for 0827-tip_finetune_onlyImage-swin_tiny_scratch_l2d192h8-tabV3-transV8-lr5e-4-warmup50-bs_64_8-neg50-ep200-tsgkf-lr1e-3-warmup5-bs_64_2-transV8-ep80 not found at /content/drive/MyDrive/isic_2024_second_place/working/configs/experiment/0827-tip_finetune_onlyImage-swin_tiny_scratch_l2d192h8-tabV3-transV8-lr5e-4-warmup50-bs_64_8-neg50-ep200-tsgkf-lr1e-3-warmup5-bs_64_2-transV8-ep80.yaml. Skipping this experiment.
Prediction files for 0825-tip_finetune_onlyImage-convnextv2_nano_scratch_l2d192h8-tabV3-transV8-lr5e-4-warmup50-bs_64_8-neg

/usr/lib/python3.12/pickle.py:1760: UserWarning: [18:50:45] WARNING: /workspace/src/collective/../data/../common/error_msg.h:82: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


,isic_id,target
0,ISIC_0015657,0.082292
1,ISIC_0015729,0.045645
2,ISIC_0015740,0.065441
